In [1]:
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import catboost as cat

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pandas as pd
import numpy as np
import pickle


In [2]:
df = pd.read_csv('start_df.csv')

df_reg = df[df['MedHouseVal'] < 5.0].copy()

x = df_reg.drop(['MedHouseVal', 'is_capped'], axis=1)
y = df_reg['MedHouseVal']

In [3]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

param_set1 = [
    {'n_estimators': 100, 'max_depth': 6},
    {'n_estimators': 150, 'max_depth': 8},
    {'n_estimators': 200, 'max_depth': 4}
]

param_set2 = [
    {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1},
    {'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.05},
    {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.2}
]


In [4]:
for params in param_set1:
    reg = RandomForestRegressor(**params)
    reg.fit(x_train, y_train)
    y_pred = reg.predict(x_test)
    print(f"Params: {params}")
    print(f"  MAE: {mean_absolute_error(y_test, y_pred):.3f}")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"  R2:  {r2_score(y_test, y_pred):.3f}\n")


Params: {'n_estimators': 100, 'max_depth': 6}
  MAE: 0.390
  RMSE: 0.545
  R2:  0.694

Params: {'n_estimators': 150, 'max_depth': 8}
  MAE: 0.344
  RMSE: 0.495
  R2:  0.747

Params: {'n_estimators': 200, 'max_depth': 4}
  MAE: 0.455
  RMSE: 0.619
  R2:  0.604



In [5]:
for params in param_set2:
    reg = xgb.XGBRegressor(**params, random_state=0)
    reg.fit(x_train, y_train)
    y_pred = reg.predict(x_test)
    print(f"Params: {params}")
    print(f"  MAE: {mean_absolute_error(y_test, y_pred):.3f}")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"  R2:  {r2_score(y_test, y_pred):.3f}\n")

Params: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1}
  MAE: 0.279
  RMSE: 0.411
  R2:  0.826

Params: {'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.05}
  MAE: 0.271
  RMSE: 0.408
  R2:  0.828

Params: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.2}
  MAE: 0.276
  RMSE: 0.408
  R2:  0.828



In [6]:
for params in param_set2:
    reg = cat.CatBoostRegressor(**params, random_seed=0, verbose=False)
    reg.fit(x_train, y_train)
    y_pred = reg.predict(x_test)
    print(f"Params: {params}")
    print(f"  MAE: {mean_absolute_error(y_test, y_pred):.3f}")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"  R2:  {r2_score(y_test, y_pred):.3f}\n")

Params: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1}
  MAE: 0.312
  RMSE: 0.450
  R2:  0.791

Params: {'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.05}
  MAE: 0.307
  RMSE: 0.445
  R2:  0.795

Params: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.2}
  MAE: 0.288
  RMSE: 0.422
  R2:  0.816



## ИТОГ

| Модель | Параметры | MAE | RMSE | R2 |
| :--- | :--- | :--- | :--- | :--- |
| RandomForest | n_estimators=150, max_depth=8 | 0.344 | 0.495 | 0.747 |
| **XGBoost** | **n_estimators=150, max_depth=6, lr=0.05** | **0.276** | **0.408** | **0.828** |
| CatBoost | iterations=200, depth=4, lr=0.2 | 0.288 | 0.422 | 0.816 |

Лучшим регрессором является XGBoost и в дальнейшем для каскада будет использован именно он. 

In [ ]:
reg = xgb.XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.05)
reg.fit(x_train, y_train)

with open('regressor_model.pkl', 'wb') as f:
    pickle.dump(reg, f)